In [1]:
import numpy as np
from scipy import stats

def ab_test_ctr(n_control, clicks_control,
                n_treat,   clicks_treat,
                alpha=0.05):
    """
    Two-proportion z-test for A/B test on ad CTR.

    Returns a dict with:
      - ctr_control  : observed CTR in control
      - ctr_treat    : observed CTR in treatment
      - lift_abs     : absolute lift (treat - control)
      - lift_rel     : relative lift as a percentage
      - z_stat       : test statistic
      - p_value      : two-sided p-value
      - ci_95        : (lower, upper) 95% CI on the absolute lift
      - reject_null  : True if p_value < alpha
      - conclusion   : string — 'New format significantly better'
                               or 'No significant difference'
    """
    # YOUR CODE HERE
    assert 0 <= clicks_control <= n_control
    assert 0 <= clicks_treat <= n_treat
    assert 0 < alpha < 1

    ctr_control = clicks_control /n_control
    ctr_treat = clicks_treat / n_treat

    lift_abs = ctr_treat -  ctr_control
    lift_rel = lift_abs/ctr_control
    
    pool_prop = (clicks_control + clicks_treat)/(n_control + n_treat)
    se = np.sqrt(pool_prop*(1-pool_prop)*(1/n_control + 1/n_treat))

    z_stat = lift_abs/se
    p_value = 2*(1-stats.norm.cdf(abs(z_stat)))

    se_unpooled = np.sqrt((ctr_control *(1-ctr_control )/n_control) + (ctr_treat* (1-ctr_treat)/n_treat))
    ci_lower = lift_abs - stats.norm.ppf(0.975)*se_unpooled
    ci_upper = lift_abs + stats.norm.ppf(0.975)*se_unpooled

    ci_95 = (ci_lower, ci_upper)

    reject_null = p_value < alpha

    if reject_null and lift_abs > 0:
        conclusion = 'New format significantly better'
    elif reject_null and lift_abs < 0:
        conclusion = 'New format significantly worse'
    else:
        conclusion = 'No significant difference'

    # ── Output validation ─────────────────────────────────
    assert 0 <= ctr_control <= 1,              "ctr_control must be in [0, 1]"
    assert 0 <= ctr_treat <= 1,                "ctr_treat must be in [0, 1]"
    assert 0 <= p_value <= 1,                  "p_value must be in [0, 1]"
    assert ci_lower < ci_upper,                "CI lower must be less than upper"
    assert reject_null == (p_value < alpha),   "reject_null must match p_value < alpha"
    assert conclusion in (
        'New format significantly better',
        'New format significantly worse',
        'No significant difference'
    ),                                         "conclusion must be one of the three valid strings"
    
    return {'ctr_control': ctr_control, 'ctr_treat': ctr_treat, 'lift_abs': lift_abs, 'lift_rel': lift_rel,
        'z_stat': z_stat ,'p_value' : p_value, 'ci_95': ci_95, 'reject_null': reject_null ,
        'conclusion': conclusion}



In [2]:

# ── Run it ──────────────────────────────────────────────
result = ab_test_ctr(
    n_control     = 80000,
    clicks_control= 1200,
    n_treat       = 80000,
    clicks_treat  = 1520,
    alpha         = 0.05
)

for k, v in result.items():
    print(f"{k:>12}: {v}")

 ctr_control: 0.015
   ctr_treat: 0.019
    lift_abs: 0.004
    lift_rel: 0.26666666666666666
      z_stat: 6.188548051217447
     p_value: 6.072089497877187e-10
       ci_95: (np.float64(0.0027333187800660267), np.float64(0.005266681219933973))
 reject_null: True
  conclusion: New format significantly better
